# Comprehensive 2D Prediction Run Analysis

This notebook provides a complete analysis of a single 2D prediction run. It loads the `prediction_stats.npz` and timeseries files generated by `gen_prediction_2d_separated.py` to visualize:

1.  **Quantitative Metrics:** Overall and per-channel R² scores.
2.  **Error Plots:** R² bar charts and error-over-time plots.
3.  **Qualitative Dashboards:** Direct visual comparisons of ground truth, predictions, and errors.

### 1. Setup and Imports

In [ ]:
from pathlib import Path
import numpy as np
import logging

# Import the new 2D evaluation plotting functions
from mhd_surrogate_core.plotting.xz import (
    plot_prediction_rollout_error,
    plot_r2_performance,
    plot_prediction_dashboard,
)

### 2. Load Evaluation and Ground Truth Data

**Action Required:** Update the paths below to point to your specific evaluation outputs.

In [ ]:
# --- DEFINE YOUR FILE PATHS HERE ---

# Path to the directory where your prediction script saved its output
eval_output_dir = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/prediction/mhd_surrogate_modelling/experiments/study_4_2d_tckae/output/no_grad_clipping/ld512_M8_K8_Ktc8_gtc1.0_bwd_False/eval/")

# Path to the original test set used for the evaluation
ground_truth_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/data/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/T1492_x1151_y1_z127_c2/preprocessed/test_set.npz")

# --- DATA LOADING LOGIC ---

# Construct full paths to the individual files
stats_file_path = eval_output_dir / "prediction_stats.npz"
predicted_file_path = eval_output_dir / "predicted_timeseries.npz"
difference_file_path = eval_output_dir / "difference_timeseries.npz"

# Check if all necessary files exist
if not all([p.exists() for p in [stats_file_path, predicted_file_path, difference_file_path, ground_truth_path]]):
    logging.error("ERROR: One or more data files not found. Please check the paths.")
else:
    # Load statistics
    with np.load(stats_file_path, allow_pickle=True) as data:
        r_squared_total = data['r_squared_total']
        channel_names = list(data['channel_names'])
        print(f"Loaded statistics from {stats_file_path}")
    
    # Load timeseries data
    with np.load(predicted_file_path, allow_pickle=True) as data:
        predicted_timeseries = data['timeseries']
        print(f"Loaded predicted timeseries from {predicted_file_path}")

    with np.load(difference_file_path, allow_pickle=True) as data:
        difference_timeseries = data['timeseries']
        print(f"Loaded difference timeseries from {difference_file_path}")
        
    with np.load(ground_truth_path, allow_pickle=True) as data:
        ground_truth_timeseries = data['timeseries']
        # Squeeze if necessary
        if ground_truth_timeseries.ndim == 5:
            ground_truth_timeseries = np.squeeze(ground_truth_timeseries, axis=2)
        coords = {
            'x': data.get('x_coords', np.arange(ground_truth_timeseries.shape[1])),
            'z': data.get('z_coords', np.arange(ground_truth_timeseries.shape[2])),
            'labels': list(data['labels'])
        }
        print(f"Loaded ground truth from {ground_truth_path}")

    # Configure max indices for dashboard widgets
    max_time_index = predicted_timeseries.shape[0] - 1
    max_x_index = predicted_timeseries.shape[1] - 1
    print(f"\nMax indices -> Time: {max_time_index}, X: {max_x_index}")

---

### 3. Quantitative Analysis

In [ ]:
print(f"--- PREDICTION SUMMARY ---")
print(f"Overall R² Score: {r_squared_total:.4f}\n")

print("\n--- Average Prediction Performance (R²) ---")
plot_r2_performance(stats_file_path, eval_type="Prediction")

print("\n--- Prediction Error Over Time ---")
plot_prediction_rollout_error(stats_file_path)

---

### 4. Qualitative Analysis Dashboard

Use this section for a detailed visual inspection of the model's performance at specific points in space and time.

In [ ]:
# === Parameters for the Dashboard ===
# <<< MODIFY THESE VALUES >>>
dashboard_channel = 'vx'       # Channel to visualize (e.g., 'vx', 'vz')
dashboard_time_index = max_time_index // 2
dashboard_x_index = max_x_index // 2
# ---

print(f"Generating dashboard for channel '{dashboard_channel}' at time index {dashboard_time_index} and x index {dashboard_x_index}")

plot_prediction_dashboard(
    ground_truth_timeseries=ground_truth_timeseries,
    predicted_timeseries=predicted_timeseries,
    difference_timeseries=difference_timeseries,
    coords=coords,
    channel=dashboard_channel,
    x_index=dashboard_x_index,
    time_index=dashboard_time_index,
)